In [ ]:
# Pré-requis à lancer avant de lancer l'extraction des cours
#!pip install yt-dlp pydub vosk python-docx
from pydub import AudioSegment
import glob
import wave
import json
from vosk import Model, KaldiRecognizer
#from docx import Document
#from docx.shared import Pt
import os
#from google.colab import files
from IPython.display import Audio, display

In [ ]:
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import torchaudio
import torch

print("All imports are successful!")


In [ ]:
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import torchaudio
import torch
audio_path = "extracted_audio.wav"
# Modèle Wav2Vec2.0 pré-entraîné
model_name = "facebook/wav2vec2-large-xlsr-53"
processor = Wav2Vec2Processor.from_pretrained(model_name)
model = Wav2Vec2ForCTC.from_pretrained(model_name)

def transcribe(file_path):
    # Charger le fichier audio
    speech, rate = torchaudio.load(file_path)
    # Préparer les entrées pour le modèle
    input_values = processor(speech, sampling_rate=rate, return_tensors="pt").input_values
    # Passer les entrées dans le modèle
    logits = model(input_values).logits
    # Décoder les prédictions
    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = processor.batch_decode(predicted_ids)
    return transcription

# Spécifier le chemin du fichier audio
audio_path = "extracted_audio.wav"
# Transcrire l'audio
print(transcribe(audio_path))


In [ ]:
import os

# Chemin vers le fichier vidéo
video_path = "video.mp4"

# Vérifier si le fichier existe
if os.path.exists(video_path):
    print("Le fichier existe.")

    # Vérifier les permissions de lecture
    if os.access(video_path, os.R_OK):
        print("Le fichier est accessible en lecture.")
    else:
        print("Le fichier n'est pas accessible en lecture. Vérifiez les permissions.")

    # Vérifier les permissions d'exécution (nécessaire pour certains types d'opérations)
    if os.access(video_path, os.X_OK):
        print("Le fichier est accessible en exécution.")
    else:
        print("Le fichier n'est pas accessible en exécution. Vérifiez les permissions.")
else:
    print("Le fichier n'existe pas à l'emplacement spécifié.")


In [ ]:
# Télécharger la vidéo YouTube
#!yt-dlp "ytsearch:'https://www.youtube.com/watch?v=RfteqehNjG8'"

In [ ]:
print("os.list")
print(os.listdir())
# Extraire l'audio de la vidéo

#video_files = glob.glob("video.*")
video_path = os.path.abspath("video.mp4") #video_files[0]

audio_path = "extracted_audio.wav"

audio = AudioSegment.from_file(video_path)
audio = audio.set_frame_rate(16000).set_channels(1)  # Convertir en mono et ajuster le taux d'échantillonnage
audio.export(audio_path, format="wav")


In [ ]:
audio = Audio(filename=audio_path)
display(audio)

In [ ]:
model_path = "vosk_model"
model = Model(model_path)

In [ ]:
# Fonction pour transcrire l'audio
import time
from IPython.display import display, clear_output


def transcribe_audio(audio_path):
    wf = wave.open(audio_path, "rb")
    rec = KaldiRecognizer(model, wf.getframerate())

    result_text = ""
    start_time = time.time()

    while True:
        elapsed_time = time.time() - start_time

        # Affichage du temps écoulé dans le notebook
        clear_output(wait=True)
        display(f"Temps écoulé: {elapsed_time:.2f} secondes")

        # Pause pour éviter d'utiliser trop de ressources CPU
        time.sleep(1)

        data = wf.readframes(4000)
        if len(data) == 0:
            break
        if rec.AcceptWaveform(data):
            result = rec.Result()
            result_text += json.loads(result).get('text', '') + " "

    final_result = rec.FinalResult()
    result_text += json.loads(final_result).get('text', '')

    return result_text

# Transcrire l'audio extrait
transcription = transcribe_audio(audio_path)
print("Transcription:\n", transcription)


In [ ]:
def download_video(url, output):
    !yt-dlp -o f'{output}.%(ext)s' {url}
    video_files = glob.glob(f"{output}.*")
    return video_files[0]

def extract_audio(video_path, audio_path):
    audio = AudioSegment.from_file(video_path)
    audio = audio.set_frame_rate(16000).set_channels(1)
    audio.export(audio_path, format="wav")

def transcribe_audio(audio_path, model_path):
    model = Model(model_path)
    wf = wave.open(audio_path, "rb")
    rec = KaldiRecognizer(model, wf.getframerate())

    result_text = ""

    while True:
        data = wf.readframes(4000)
        if len(data) == 0:
            break
        if rec.AcceptWaveform(data):
            result = rec.Result()
            result_text += json.loads(result).get('text', '') + " "

    final_result = rec.FinalResult()
    result_text += json.loads(final_result).get('text', '')

    return result_text

def create_document(transcription, doc_output):
    doc = Document()
    doc.add_heading('Transcription', 0)

    content = transcription.split('. ')
    for line in content:
        add_paragraph(doc, line)

    doc.save(doc_output)

def add_paragraph(doc, text, style=None):
    paragraph = doc.add_paragraph(text)
    if style:
        paragraph.style = style
    paragraph.style.font.size = Pt(12)
    return paragraph


In [ ]:
    url = 'https://www.youtube.com/watch?v=RfteqehNjG8'
    video_output = 'video'
    audio_output = 'extracted_audio.wav'
    model_url = 'https://alphacephei.com/vosk/models/vosk-model-small-fr-0.22.zip'
    model_dir = 'vosk_model'
    model_path = f'{model_dir}/vosk-model-small-fr-0.22'


In [ ]:
# Télécharger la vidéo YouTube
video_path = download_video(url, video_output)

# Extraire l'audio de la vidéo
extract_audio(video_path, audio_output)

# Télécharger et charger le modèle Vosk pour le français
!wget -O model-fr.zip {model_url}
!unzip model-fr.zip -d {model_dir}

In [ ]:
# Lecture de l'audio
audio = Audio(filename=audio_output)
display(audio)


In [ ]:
transcription = transcribe_audio(audio_output, model_path)
# Créer un document Word avec la transcription
#doc_output = "Transcription.docx"
#create_document(transcription, doc_output)

# Télécharger le document
#files.download(doc_output)
